## USING LSTM TO PREDICT VAR

### Setting up data and sets

In [66]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# 1. Load the data 
# (Added index_col=0 so your dates stay in the index, not as a separate column)
path = r"C:\Users\nisha\Desktop\2026 PS-II\Projects\VaR\pythonProject1\data"
import_file = os.path.join(path, 'Data_with_features.csv')
sp500_data = pd.read_csv(import_file, index_col=0, parse_dates=True)

# 2. Isolate features 
features = ['Return', 'Rolling_GARCH_Vol', 'RiskMetrics_Vol', 'VIX_Daily_Shifted', 'RSI']

lstm_data = sp500_data[features].copy()
lstm_data.dropna(inplace=True)

# Calculate where the 80% split happens in the raw dataframe
split_row = int(len(lstm_data) * 0.8)
train_subset = lstm_data.iloc[:split_row]

# Initialize and FIT the scaler ONLY on the training subset
scaler = MinMaxScaler(feature_range=(0, 1))
scaler.fit(train_subset)

# Now apply that transformation scale to the entire dataset
scaled_data = scaler.transform(lstm_data)

# 4. Create Sequences (X: Past 60 days -> Y: Current day Return)
X = []
y = []
window_size = 60

for i in range(window_size, len(scaled_data)):
    X.append(scaled_data[i-window_size : i, :])
    y.append(scaled_data[i, 0])  # 0 is the column index for 'Return'
    
X, y = np.array(X), np.array(y) 

# 5. Split into Train and Test (80/20)
split_idx = int(len(X) * 0.8)

X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(f"X_train shape: {X_train.shape} (Samples, Timesteps, Features)")
print(f"X_test shape:  {X_test.shape} (Samples, Timesteps, Features)")

X_train shape: (4967, 60, 5) (Samples, Timesteps, Features)
X_test shape:  (1242, 60, 5) (Samples, Timesteps, Features)


### Loss Function

In [67]:
import tensorflow as tf

# Set the VaR confidence level 
target_alpha = 0.02

# The Core Pinball/Quantile Loss Function
def quantile_loss(q, y_true, y_pred):
    error = y_true - y_pred
    return tf.reduce_mean(tf.maximum(q * error, (q - 1) * error))

# Clean Wrapper Function for Keras Compilation
def target_quantile_loss(y_true, y_pred):
    return quantile_loss(target_alpha, y_true, y_pred)

### Model

In [68]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam

# Initialize Model
model = Sequential()

# Layer 1: LSTM (Outputs sequences so the next LSTM can read them)
model.add(Input(shape=(X_train.shape[1], X_train.shape[2])))  # ← Fixed: (timesteps, features)
model.add(LSTM(units=64, return_sequences=True, activation='tanh'))
model.add(Dropout(0.2)) 

# Layer 2: LSTM (Compresses the sequence into a final hidden state)
model.add(LSTM(units=32, activation='tanh'))
model.add(Dropout(0.2))

# Layer 3: Output (1 Neuron = The predicted VaR boundary)
model.add(Dense(units=1))

# Compile with our Custom Quantile Loss
optimizer = Adam(learning_rate=0.001)
model.compile(optimizer=optimizer, loss=target_quantile_loss)

# Display the architecture
model.summary()

Model: "sequential_12"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_14 (LSTM)                  │ (None, 60, 64)         │        17,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_14 (Dropout)            │ (None, 60, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_15 (LSTM)                  │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_15 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 30,369 (118.63 KB)

 Trainable params: 30,369 (118.63 KB)

 Non-trainable params: 0 (0.00 B)

### Training


In [69]:
# Train the LSTM network for Value-at-Risk forecasting
history = model.fit(
    X_train, y_train,
    epochs=20,           
    batch_size=64,       
    validation_data=(X_test, y_test),
    verbose=1
)

Epoch 1/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 6s 44ms/step - loss: 0.0049 - val_loss: 0.0031
Epoch 2/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 3s 38ms/step - loss: 0.0037 - val_loss: 0.0026
Epoch 3/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 3s 38ms/step - loss: 0.0038 - val_loss: 0.0027
Epoch 4/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 3s 40ms/step - loss: 0.0035 - val_loss: 0.0029
Epoch 5/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 3s 38ms/step - loss: 0.0034 - val_loss: 0.0028
Epoch 6/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 3s 35ms/step - loss: 0.0033 - val_loss: 0.0034
Epoch 7/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 3s 36ms/step - loss: 0.0032 - val_loss: 0.0035
Epoch 8/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 3s 37ms/step - loss: 0.0034 - val_loss: 0.0029
Epoch 9/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 3s 36ms/step - loss: 0.0033 - val_loss: 0.0030
Epoch 10/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 3s 36ms/step - loss: 0.0031 - val_loss: 0.0026
Epoch 11/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 3s 38ms/step - loss: 0.0031 - val_loss: 0.0028
Epoch 12/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 3s 37ms/step - loss: 0.0

In [70]:
# 1. Generate scaled predictions
lstm_pred_scaled = model.predict(X, verbose=0)

# 2. Inverse Transform (The Dummy Matrix Trick)
# Create a 5-column matrix of zeros to satisfy the scaler's shape requirement
dummy_matrix = np.zeros((len(lstm_pred_scaled), 5))

# Inject the predictions into the first column (Index 0 = 'Return')
dummy_matrix[:, 0] = lstm_pred_scaled.flatten()

# Inverse transform and extract the un-scaled real values
lstm_pred_real = scaler.inverse_transform(dummy_matrix)[:, 0]

# 3. Assign to DataFrame cleanly
sp500_data['LSTM_VaR'] = np.nan
sp500_data.loc[sp500_data.index[window_size:], 'LSTM_VaR'] = lstm_pred_real

# 4. Verify the alignment and scale
print(sp500_data[['Return', 'LSTM_VaR']])
export_file = os.path.join(path, 'Results.csv')
sp500_data.to_csv(export_file)

              Return  LSTM_VaR
Date                          
2001-05-16  2.805552       NaN
2001-05-17  0.272005       NaN
2001-05-18  0.268943       NaN
2001-05-21  1.602466       NaN
2001-05-22 -0.263133       NaN
...              ...       ...
2026-04-15  0.794414 -2.562281
2026-04-16  0.260656 -2.463645
2026-04-17  1.196855 -2.383947
2026-04-20 -0.237720 -2.287802
2026-04-21 -0.636845 -2.234172

[6269 rows x 2 columns]
